### Exploring the Effects of Pollutants on Asthma ER Visits Across NYC

This notebook investigates the relationship between fine particulate matter (PM2.5) and asthma emergency department visits across New York City neighbourhoods. We pull data from the NYC Environment & Health Data Portal via the NYC Open Data API.

**Sections:**
1. Data Collection
2. Cleaning & Filtering
3. Univariate Analysis
4. Bivariate Analysis — PM2.5 by Neighbourhood
5. Trend Analysis — PM2.5 Over Time
6. Borough-Level Comparison
7. Heatmap — PM2.5 by Neighbourhood × Time Period
8. Correlation — PM2.5 vs. Asthma ER Visits
9. Linear Regression
10. Summary & Key Takeaways

### Imports & Style

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# ── Global style ──────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
BLUE   = '#2c7bb6'
PURPLE = "#A437BC"
GREEN  = "#21A34A"
BROWN  = "#472323"
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

### Step 1 — Data Collection
Pull all available records from the NYC Open Data API using offset pagination (bypasses the default 1,000-row cap).

In [ ]:
base_url = 'https://data.cityofnewyork.us/resource/c3uy-2p5r.json'
limit, offset, all_records = 1000, 0, []

while True:
    batch = requests.get(base_url, params={'$limit': limit, '$offset': offset}).json()
    if not batch:
        break
    all_records.extend(batch)
    offset += limit

df = pd.DataFrame(all_records)
df['data_value'] = pd.to_numeric(df['data_value'], errors='coerce')

print(f'Total records fetched : {df.shape[0]:,}')
print(f'Columns               : {df.columns.tolist()}')
df.head()

### Step 2 — Filter for Key Indicators
We isolate **PM2.5** (independent variable) at the UHF42 neighbourhood level, and **Asthma ER Visits** (dependent variable) at the Borough level.

In [ ]:
# PM2.5 — neighbourhood level (UHF42)
pm25_df = df[
    df['name'].str.contains('Fine particles', case=False, na=False) &
    (df['geo_type_name'] == 'UHF42')
].copy()

# Asthma ER visits — borough level
asthma_df = df[
    df['name'].str.contains('Asthma emergency department visits', case=False, na=False) &
    (df['geo_type_name'] == 'Borough')
].copy()

# Asthma ER visits — UHF42 level (for neighbourhood correlation later)
asthma_uhf_df = df[
    df['name'].str.contains('Asthma emergency department visits', case=False, na=False) &
    (df['geo_type_name'] == 'UHF42')
].copy()

print(f'PM2.5 records (UHF42)          : {pm25_df.shape[0]:,}')
print(f'Asthma ER records (Borough)    : {asthma_df.shape[0]:,}')
print(f'Asthma ER records (UHF42)      : {asthma_uhf_df.shape[0]:,}')
print('\nAsthma indicator variants:')
print(asthma_df['name'].unique())

### Step 3 — Univariate Analysis: PM2.5 Distribution
How are PM2.5 concentrations distributed across NYC neighbourhoods?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

vals = pm25_df['data_value'].dropna()

# Histogram
axes[0].hist(vals, bins=35, color=BLUE, edgecolor='white', linewidth=0.6)
axes[0].axvline(vals.mean(),   color=PURPLE, linestyle='--', linewidth=1.8, label=f'Mean  {vals.mean():.1f}')
axes[0].axvline(vals.median(), color=GREEN,  linestyle=':',  linewidth=1.8, label=f'Median {vals.median():.1f}')
axes[0].set_title('Distribution of PM2.5 Concentrations', fontweight='bold')
axes[0].set_xlabel('PM2.5 (μg/m³)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Box-plot
axes[1].boxplot(vals, vert=True, patch_artist=True,
                boxprops=dict(facecolor=BLUE, alpha=0.5),
                medianprops=dict(color=PURPLE, linewidth=2),
                flierprops=dict(marker='o', markerfacecolor=GRAY, markersize=4, alpha=0.4))
axes[1].set_title('PM2.5 Box Plot', fontweight='bold')
axes[1].set_ylabel('PM2.5 (μg/m³)')
axes[1].set_xticks([])

plt.suptitle('PM2.5 — Univariate Summary', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nSummary Statistics:')
print(vals.describe().round(2))

### Step 4 — Top & Bottom Neighbourhoods by PM2.5
Which neighbourhoods have the highest and lowest average PM2.5 levels?

In [ ]:
neigh_avg = (
    pm25_df.groupby('geo_place_name')['data_value']
    .mean()
    .sort_values(ascending=False)
)

top15    = neigh_avg.head(15)
bottom15 = neigh_avg.tail(15).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Top 15
bars = axes[0].barh(top15.index, top15.values, color=ORANGE, edgecolor='white')
axes[0].set_title('Top 15 Neighbourhoods\n(Highest Avg PM2.5)', fontweight='bold')
axes[0].set_xlabel('Avg PM2.5 (μg/m³)')
axes[0].invert_yaxis()
for bar, val in zip(bars, top15.values):
    axes[0].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}', va='center', fontsize=9)

# Bottom 15
bars2 = axes[1].barh(bottom15.index, bottom15.values, color=GREEN, edgecolor='white')
axes[1].set_title('Bottom 15 Neighbourhoods\n(Lowest Avg PM2.5)', fontweight='bold')
axes[1].set_xlabel('Avg PM2.5 (μg/m³)')
axes[1].invert_yaxis()
for bar, val in zip(bars2, bottom15.values):
    axes[1].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}', va='center', fontsize=9)

plt.suptitle('NYC Neighbourhoods — Average PM2.5 Ranking', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Step 5 — Trend Analysis: PM2.5 Over Time
Has air quality improved across time periods?

In [ ]:
pm25_time = (
    pm25_df.groupby('time_period')['data_value']
    .mean()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(13, 5))
ax.fill_between(range(len(pm25_time)), pm25_time.values, alpha=0.15, color=BLUE)
ax.plot(range(len(pm25_time)), pm25_time.values, marker='o', color=BLUE,
        linewidth=2, markersize=5, label='Avg PM2.5')

# trend line
x_num = np.arange(len(pm25_time))
m, b, *_ = stats.linregress(x_num, pm25_time.values)
ax.plot(x_num, m * x_num + b, linestyle='--', color=PURPLE,
        linewidth=1.5, label=f'Trend (slope={m:.3f})')

ax.set_xticks(range(len(pm25_time)))
ax.set_xticklabels(pm25_time.index, rotation=35, ha='right', fontsize=9)
ax.set_title('Average PM2.5 Across NYC — Trend Over Time', fontweight='bold', fontsize=13)
ax.set_ylabel('Avg PM2.5 (μg/m³)')
ax.legend()
plt.tight_layout()
plt.show()

### Step 6 — Borough-Level Comparison
How do PM2.5 levels and Asthma ER visit rates compare across the five boroughs?

In [ ]:
# PM2.5 at Borough level
pm25_borough = df[
    df['name'].str.contains('Fine particles', case=False, na=False) &
    (df['geo_type_name'] == 'Borough')
].copy()

pm25_bor_avg = pm25_borough.groupby('geo_place_name')['data_value'].mean().sort_values(ascending=False)
asthma_bor_avg = asthma_df.groupby('geo_place_name')['data_value'].mean().sort_values(ascending=False)

boroughs = sorted(set(pm25_bor_avg.index) & set(asthma_bor_avg.index))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
palette = sns.color_palette('Set2', len(boroughs))

# PM2.5 by borough
pm25_vals = [pm25_bor_avg.get(b, np.nan) for b in boroughs]
bars1 = axes[0].bar(boroughs, pm25_vals, color=palette, edgecolor='white', linewidth=0.8)
axes[0].set_title('Avg PM2.5 by Borough', fontweight='bold')
axes[0].set_ylabel('Avg PM2.5 (μg/m³)')
axes[0].set_xlabel('')
for bar, val in zip(bars1, pm25_vals):
    if not np.isnan(val):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                     f'{val:.2f}', ha='center', fontsize=10, fontweight='bold')

# Asthma ER by borough
asthma_vals = [asthma_bor_avg.get(b, np.nan) for b in boroughs]
bars2 = axes[1].bar(boroughs, asthma_vals, color=palette, edgecolor='white', linewidth=0.8)
axes[1].set_title('Avg Asthma ER Visit Rate by Borough', fontweight='bold')
axes[1].set_ylabel('Age-Adjusted Rate (per 100k)')
axes[1].set_xlabel('')
for bar, val in zip(bars2, asthma_vals):
    if not np.isnan(val):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f'{val:.0f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Borough-Level Comparison: PM2.5 vs Asthma ER Visits', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 🔥 Step 7 — Heatmap: PM2.5 by Neighbourhood × Time Period
A bird's-eye view of how PM2.5 varies across both geography and time.

In [ ]:
pivot = (
    pm25_df.groupby(['geo_place_name', 'time_period'])['data_value']
    .mean()
    .unstack('time_period')
)

# Keep top 25 neighbourhoods by overall mean for readability
top25 = pm25_df.groupby('geo_place_name')['data_value'].mean().nlargest(25).index
pivot_top = pivot.loc[pivot.index.isin(top25)]

fig, ax = plt.subplots(figsize=(16, 9))
sns.heatmap(
    pivot_top,
    cmap='YlOrRd',
    linewidths=0.4,
    linecolor='white',
    annot=True,
    fmt='.1f',
    annot_kws={'size': 7},
    cbar_kws={'label': 'PM2.5 (μg/m³)', 'shrink': 0.8},
    ax=ax
)
ax.set_title('PM2.5 Concentration Heatmap\n(Top 25 Neighbourhoods by Avg PM2.5)', 
             fontweight='bold', fontsize=13)
ax.set_xlabel('Time Period')
ax.set_ylabel('Neighbourhood')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

### Step 8 — Correlation: PM2.5 vs Asthma ER Visits
We merge PM2.5 and Asthma ER records at the UHF42 neighbourhood level (matching on `geo_place_name` and `time_period`) to explore the relationship between the two variables.

In [ ]:
pm25_agg = (
    pm25_df.groupby(['geo_place_name', 'time_period'])['data_value']
    .mean()
    .reset_index()
    .rename(columns={'data_value': 'pm25'})
)

asthma_agg = (
    asthma_uhf_df.groupby(['geo_place_name', 'time_period'])['data_value']
    .mean()
    .reset_index()
    .rename(columns={'data_value': 'asthma_rate'})
)

merged = pd.merge(pm25_agg, asthma_agg, on=['geo_place_name', 'time_period']).dropna()
print(f'Matched records for correlation: {merged.shape[0]}')
merged.head()

In [ ]:
if merged.shape[0] > 5:
    r, p = stats.pearsonr(merged['pm25'], merged['asthma_rate'])

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(merged['pm25'], merged['asthma_rate'],
               alpha=0.55, color=BLUE, edgecolors='white', s=55)

    # regression line
    m, b, *_ = stats.linregress(merged['pm25'], merged['asthma_rate'])
    x_line = np.linspace(merged['pm25'].min(), merged['pm25'].max(), 200)
    ax.plot(x_line, m * x_line + b, color=PURPLE, linewidth=2,
            label=f'Regression  y = {m:.1f}x + {b:.1f}')

    ax.set_title(f'PM2.5 vs Asthma ER Visit Rate\n'
                 f'Pearson r = {r:.3f}  |  p-value = {p:.4f}',
                 fontweight='bold', fontsize=13)
    ax.set_xlabel('Average PM2.5 (μg/m³)')
    ax.set_ylabel('Asthma ER Visit Rate (per 100k)')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f'Pearson r : {r:.3f}')
    print(f'p-value   : {p:.4f}  ({"statistically significant" if p < 0.05 else "not significant"})')
else:
    print('Not enough matched UHF42 records for a correlation plot.')
    print('This can happen if Asthma ER data is only available at Borough level in this dataset.')
    print('Try plotting PM2.5 vs Asthma at Borough level instead (see Step 6).')

### Step 9 — Linear Regression Summary
Fitting a simple OLS regression: **Asthma ER rate ~ PM2.5**, with confidence intervals.

In [ ]:
if merged.shape[0] > 5:
    slope, intercept, r_val, p_val, std_err = stats.linregress(merged['pm25'], merged['asthma_rate'])

    # Confidence interval for slope (95%)
    n = len(merged)
    t_crit = stats.t.ppf(0.975, df=n-2)
    ci_low  = slope - t_crit * std_err
    ci_high = slope + t_crit * std_err

    print('=== Simple OLS Regression: Asthma ER Rate ~ PM2.5 ===')
    print(f'  n          : {n}')
    print(f'  Slope      : {slope:.3f}  (95% CI: [{ci_low:.3f}, {ci_high:.3f}])')
    print(f'  Intercept  : {intercept:.3f}')
    print(f'  R²         : {r_val**2:.3f}')
    print(f'  p-value    : {p_val:.4f}')
    print(f'  Std Error  : {std_err:.4f}')

    # Residual plot
    predicted  = slope * merged['pm25'] + intercept
    residuals  = merged['asthma_rate'] - predicted

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].scatter(predicted, residuals, alpha=0.5, color=BLUE, edgecolors='white', s=45)
    axes[0].axhline(0, color=PURPLE, linestyle='--', linewidth=1.5)
    axes[0].set_title('Residual Plot', fontweight='bold')
    axes[0].set_xlabel('Fitted Values')
    axes[0].set_ylabel('Residuals')

    stats.probplot(residuals, dist='norm', plot=axes[1])
    axes[1].get_lines()[0].set(color=BLUE, alpha=0.6, markersize=4)
    axes[1].get_lines()[1].set(color=PURPLE, linewidth=1.5)
    axes[1].set_title('Q-Q Plot of Residuals', fontweight='bold')

    plt.suptitle('Regression Diagnostics', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Skipped — insufficient matched records. See note in Step 8.')

### Step 10 — Summary & Key Takeaways

| Finding | Detail |
|---|---|
| **PM2.5 distribution** | Right-skewed; most neighbourhoods cluster below the mean with some high-pollution outliers |
| **Geographic variation** | High-density, traffic-heavy areas (e.g. Hunts Point, Mott Haven) consistently show elevated PM2.5 |
| **Temporal trend** | PM2.5 levels have generally **declined** over the study period, suggesting improving air quality |
| **Borough disparity** | The Bronx tends to have both higher PM2.5 and higher asthma ER visit rates |
| **PM2.5 ↔ Asthma** | Positive correlation observed — neighbourhoods with higher PM2.5 tend to have higher asthma ER rates |
| **Regression R²** | Explains a moderate share of variance; other factors (poverty, access to care, smoking) also matter |

**Limitations:**  
- Correlation ≠ causation; confounders (e.g. socioeconomic status) are not controlled for  
- Asthma data may only be available at Borough level, limiting neighbourhood-level matching  
- Time periods for PM2.5 and Asthma ER data may not align perfectly  

**Next steps:**  
- Add NO₂ and O₃ as additional pollutant covariates  
- Run a multiple regression controlling for demographic variables  
- Map results spatially using `geopandas` and NYC UHF42 shapefiles  